# Uncertainty Quantification with PET-MAD

*A fill-in-the-blank exercise notebook.* Adapted from the
[atomistic cookbook](https://atomistic-cookbook.org) recipe *Uncertainty Quantification
with PET-MAD* by Johannes Spies, Filippo Bigi, and Matthias Kellner.

You will compute errors on the outputs of ML-potential-driven simulations using
**PET-MAD** and its built-in uncertainty quantification (UQ), in three steps:

1. **Uncertainties on a dataset** — single-point energy uncertainties on a validation
   set, comparing the analytical *LLPR* uncertainty with the *last-layer ensemble* spread.
2. **Vacancy formation energies** — propagating ensemble uncertainty to a simple derived
   quantity.
3. **Uncertainty propagation in MD** — propagating errors to a thermodynamic average
   (a radial distribution function), both with a hand-written reweighting and with i-PI.


## Setup and imports

The first run downloads the PET-MAD model (and, if missing, the validation data), so an
internet connection is needed the first time. Everything runs on CPU.

In [1]:
import os
import subprocess

import upet

import ase.io
import ase.geometry.rdf
import ase.units

import chemiscope
import matplotlib.pyplot as plt

import numpy as np
import torch

from ase import Atoms
from ase.filters import FrechetCellFilter
from ase.io.cif import read_cif
from ase.md.bussi import Bussi
from ase.md.velocitydistribution import (
    MaxwellBoltzmannDistribution,
    Stationary,
    ZeroRotation,
)
from ase.optimize.bfgs import BFGS

from ipi.utils.scripting import InteractiveSimulation
from metatomic.torch import ModelOutput
from metatomic_ase import MetatomicCalculator
from metatrain.utils.data import Dataset, read_systems, read_targets
from metatrain.utils.data.system_to_ase import system_to_ase

/var/folders/7y/yhl65v0j5m3g8zj_xwkzq44w0000gn/T/ipykernel_73531/1409548783.py:27: DeprecationWarning: ipi.utils.scripting has moved to ipi.scripting; update your imports to `from ipi.scripting import ...`. This compatibility shim will be removed in a future release.
  from ipi.utils.scripting import InteractiveSimulation


## Model loading

All three exercises need a PET-MAD model with both *ensemble* and *LLPR* uncertainty
prediction. PET-MAD v1.5.0 ships with built-in UQ, so we use it directly. We take the
extra-small (`xs`) model for speed; for production the small (`s`) model is far more
accurate and still fast. We download it with `upet` and wrap it in the ASE-compatible
`MetatomicCalculator` (which also handles the neighbor lists).

In [2]:


ckpt_path = "/home/max/cosmo_models/pet-mad-xs-v1.5.0.ckpt"
model_store_path = "models/pet-mad-xs-v1.5.0.pt"

os.makedirs("models", exist_ok=True)

upet.save_upet(checkpoint_path=ckpt_path, output=model_store_path)

calculator = MetatomicCalculator(model_store_path, device="cpu")

RuntimeError: open file failed because of errno 2 on fopen: No such file or directory, file path: /home/max/cosmo_models/pet-mad-xs-v1.5.0.ckpt

## Exercise 1 — Uncertainties on a dataset

We estimate uncertainties on a reference dataset: a reduced subset (100 structures) of
the MAD-1.5 validation set, whose r²SCAN references match the level of theory PET-MAD
v1.5.0 was trained on. The full set comes from
[Materials Cloud](https://archive.materialscloud.org/records/18tke-tt476); a local
100-structure copy is shipped in `data/`, so the download step is skipped when present.

In [3]:
# Read the dataset's structures.
systems = read_systems("data/mad-val-100.xyz")

# Read the dataset's targets (the reference atomization energies).
target_config = {
    "energy": {
        "quantity": "energy",
        "read_from": "data/mad-val-100.xyz",
        "reader": "ase",
        "key": "atomization_energy",
        "unit": "eV",
        "type": "scalar",
        "per_atom": False,
        "num_subtargets": 1,
        "forces": False,
        "stress": False,
        "virial": False,
    },
}
targets, infos = read_targets(target_config)

# Wrap in a `metatrain`-compatible dataset.
dataset = Dataset.from_dict({"system": systems, **targets})

### Exercise 1a — request the uncertainty outputs

PET-MAD can report two complementary uncertainties, but **only if you ask for them**:

- `energy_uncertainty` — the analytical **LLPR** uncertainty, and
- `energy_ensemble` — the predictions of a **last-layer ensemble** (one energy per member).

**Your task.** Add both keys to the `outputs` request below.
*Hint:* each requested output is just `ModelOutput()`, keyed by its name.

*(The collapsed `…` cell right after holds the reference solution.)*

In [ ]:
# Convert the systems to ASE `Atoms` objects.
systems = [system_to_ase(sample["system"]) for sample in dataset]

# TODO (Exercise 1a): request the two UQ outputs as well.
outputs = {
    "energy": ModelOutput(),  # the energy prediction
    # "energy_uncertainty": ModelOutput(),   # <-- add this line
    # "energy_ensemble": ModelOutput(),      # <-- and this one
}

In [ ]:
# Reference solution (Exercise 1a): make sure both UQ outputs are requested.
for _needed in ("energy_uncertainty", "energy_ensemble"):
    outputs.setdefault(_needed, ModelOutput())

In [ ]:
# Run the model and pull out the predictions.
results = calculator.run_model(systems, outputs)

predicted_energies = results["energy"][0].values.squeeze()
predicted_uncertainties = results["energy_uncertainty"][0].values.squeeze()
ensemble_raw = results["energy_ensemble"][0].values
print("energy_ensemble shape:", ensemble_raw.shape)  # (n_structures, n_members)

### Exercise 1b — the ensemble uncertainty

`ensemble_raw` has shape `(n_structures, n_members)`. The ensemble uncertainty of a
structure is simply how much its members **disagree** — the standard deviation across the
members.

**Your task.** Set `predicted_ensemble_std` to the per-structure standard deviation over
the last axis of `ensemble_raw`.
*Hint:* `tensor.std(dim=-1)`; you may need `.squeeze()` to drop a trailing length-1 axis.

In [ ]:
# TODO (Exercise 1b): standard deviation across the ensemble members (last axis).
predicted_ensemble_std = None  # <-- your code, e.g. ensemble_raw.std(dim=-1)...

In [ ]:
# Reference solution (Exercise 1b).
if predicted_ensemble_std is None:
    predicted_ensemble_std = ensemble_raw.std(dim=-1).squeeze()

Finally we need the *true* error to compare against — the absolute difference
between the predicted energy and the dataset reference.

In [ ]:
# Reference values from the dataset.
ground_truth_energies = torch.stack(
    [sample["energy"][0].values.squeeze() for sample in dataset]
)

# Absolute error between predicted energy and reference value.
empirical_errors = torch.abs(predicted_energies - ground_truth_energies)

We compare predicted uncertainties against the true errors in a **parity plot**
(cf. Fig. S4 of the PET-MAD paper; see Appendix F.7 of
[Bigi *et al.*, 2024](https://arxiv.org/abs/2403.02251) for how to read it). The two
panels put the calibrated LLPR uncertainty next to the ensemble standard deviation.

**What to look for.** Both axes are logarithmic. Well-calibrated uncertainties scatter
*around* the diagonal (the dashed line); the dotted iso-lines mark fixed error/uncertainty
ratios. With only 100 structures the plots look sparse.

In [ ]:
quantile_lines = [0.00916, 0.10256, 0.4309805, 1.71796, 2.5348, 3.44388]
_all_data = torch.cat(
    [predicted_uncertainties, predicted_ensemble_std, empirical_errors]
)
min_val = float(_all_data[_all_data > 0].min()) * 0.5
max_val = float(_all_data.max()) * 2.0

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, uncertainty, title, xlabel in [
    (axes[0], predicted_uncertainties, "LLPR", "Predicted energy uncertainty / eV"),
    (axes[1], predicted_ensemble_std, "Ensemble", "Ensemble standard deviation / eV"),
]:
    ax.set(xscale="log", yscale="log", title=title, xlabel=xlabel)
    ax.grid()
    ax.plot([min_val, max_val], [min_val, max_val], "k--")
    for factor in quantile_lines:
        ax.plot([min_val, max_val], [factor * min_val, factor * max_val], "k:", lw=0.75)
    ax.scatter(uncertainty, empirical_errors, s=10)

axes[0].set_ylabel("Absolute energy error / eV")
fig.tight_layout()
plt.show()

### Visualizing uncertainties with chemiscope

We use [chemiscope](https://chemiscope.org/docs/) to interactively explore which
structures have larger errors or higher uncertainty. We pass the (logarithms of the) same
uncertainty and error values shown in the parity plot — click points in the map to see
the corresponding structure.

In [ ]:
chemiscope.show(
    systems,
    properties={
        "ln(LLPR uncertainty / eV)": np.log(predicted_uncertainties.detach().numpy()),
        "ln(Ensemble std / eV)": np.log(predicted_ensemble_std.detach().numpy()),
        "ln(Empirical error / eV)": np.log(empirical_errors.detach().numpy()),
    },
    settings={
        "map": {
            "x": {"property": "ln(LLPR uncertainty / eV)"},
            "y": {"property": "ln(Empirical error / eV)"},
            "color": {
                "property": "ln(Ensemble std / eV)",
                "scheme": "inferno",
                "log": True,
            },
        },
        "structure": [
            {"playbackDelay": 50, "unitCell": True, "bonds": True, "spaceFilling": True}
        ],
    },
)

## Exercise 2 — Uncertainties in vacancy formation energies

Ensemble UQ also lets us estimate the error on a
[vacancy formation](https://en.wikipedia.org/wiki/Vacancy_defect) energy — propagating
uncertainty through a simple *function* of energies. We use an aluminum crystal
(downloaded from the
[Materials Project](https://legacy.materialsproject.org/materials/mp-134/) and shipped in
`data/`), build a supercell, and track the ensemble energy before the defect, right after
removing an atom, and after relaxing the defected cell.

In [ ]:
# Load the crystal and create a supercell (not strictly necessary).
crystal_structure = "data/Al_mp-134_conventional_standard.cif"
atoms = read_cif(crystal_structure)
supercell = atoms * 2
supercell.calc = calculator
N = len(supercell)  # number of atoms in the pristine supercell

We keep the ensemble energies at each stage. Calling `run_model` with the
`energy_ensemble` output returns one energy per ensemble member.

In [ ]:
# Get ensemble energy before creating the vacancy.
outputs = ["energy", "energy_ensemble"]
outputs = {o: ModelOutput() for o in outputs}
results = calculator.run_model(supercell, outputs)
bulk = results["energy_ensemble"][0].values

# Remove an atom (the last one) to create a vacancy.
i = -1
supercell.pop(i)

# Ensemble energy right after creating the vacancy (unrelaxed).
results = calculator.run_model(supercell, outputs)
right_after_vacancy = results["energy_ensemble"][0].values

# Relax both atomic positions and the cell.
ecf = FrechetCellFilter(supercell)
bfgs = BFGS(ecf)
bfgs.run()

# Ensemble energy of the relaxed vacancy.
results = calculator.run_model(supercell, outputs)
vacancy = results["energy_ensemble"][0].values

### Exercise 2 — the formation energy

The vacancy formation energy compares the relaxed defected cell (`N - 1` atoms) with the
same number of atoms taken from the pristine bulk (`N` atoms):

$$E_\text{form} = E_\text{vacancy} - \frac{N-1}{N}\, E_\text{bulk}$$

**Your task.** Compute `vacancy_formation` *for each ensemble member*. Because `vacancy`
and `bulk` are per-member tensors, the formula is one vectorized line that yields one
formation energy per member.
*Hint:* translate the equation directly, using `vacancy`, `bulk`, and `N`.

In [ ]:
# TODO (Exercise 2): vacancy formation energy per ensemble member.
vacancy_formation = None  # <-- your code: E_vacancy - (N - 1) / N * E_bulk

In [ ]:
# Reference solution (Exercise 2).
if vacancy_formation is None:
    vacancy_formation = vacancy - (N - 1) / N * bulk

In [ ]:
# For comparison, the *unrelaxed* formation energy uses the same formula.
unrelaxed_formation = right_after_vacancy - (N - 1) / N * bulk

Because every member gives its own value, we get a mean **and** a standard
deviation for each quantity.

**What to look for.** The *total* energies have a large spread, yet the formation energy
spread is much smaller — the per-member bias largely **cancels** in the difference. This
error cancellation is exactly why differences of ML energies are often far more reliable
than the absolute energies themselves.

In [ ]:
defect_energy_samples = [
    ("Bulk energy", bulk.detach().numpy().squeeze()),
    ("Unrelaxed vacancy energy", right_after_vacancy.detach().numpy().squeeze()),
    ("Relaxed vacancy energy", vacancy.detach().numpy().squeeze()),
    ("Unrelaxed VFE", unrelaxed_formation.detach().numpy().squeeze()),
    ("Relaxed VFE", vacancy_formation.detach().numpy().squeeze()),
]

for name, values in defect_energy_samples:
    print(f"{name}: mean = {values.mean():.6f} eV, std = {values.std(ddof=1):.6f} eV")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
defect_means = np.array([values.mean() for _, values in defect_energy_samples])
defect_stds = np.array([values.std(ddof=1) for _, values in defect_energy_samples])

for ax, selection, title in [
    (axes[0], slice(0, 3), "Total energies"),
    (axes[1], slice(3, 5), "Formation energies"),
]:
    x = np.arange(len(defect_energy_samples[selection]))
    ax.errorbar(
        x,
        defect_means[selection],
        yerr=defect_stds[selection],
        fmt="o",
        capsize=4,
    )
    ax.set(
        xticks=x,
        xticklabels=[
            label.replace(" ", "\n") for label, _ in defect_energy_samples[selection]
        ],
        title=title,
    )
    ax.grid(axis="y")

fig.supylabel("Energy / eV")
fig.tight_layout()
plt.show()

## Exercise 3 — Uncertainty propagation in MD

We now propagate ensemble uncertainty to a thermodynamic observable. We run a short NVT
simulation of 32 water molecules, evaluate the ensemble energy of each sampled frame, and
reweight every frame for each ensemble member. The observable is the O–H
[radial distribution function](https://en.wikipedia.org/wiki/Radial_distribution_function)
(RDF).

Computing the RDF uncertainty analytically (linearization / Gaussian error propagation)
is infeasible: the RDF is a highly non-linear functional of the positions. It is realistically also 
infeasible to run molecular dynamics simulations for each committee member - after all we cannot afford to run 128 molecular dynamics simulations, each for a different member.
Instead we run
**one** trajectory under the mean potential $\bar{V}$ and reweight each frame in
post-processing to estimate the observable-averaged had they been computed using committee members $V^{(i)}$ — the spread across all
members is the uncertainty.

In [ ]:
temperature_K = 350.0
water_md = ase.io.read("data/h2o-32.xyz")
water_md.set_cell([9.865916, 9.865916, 9.865916])
water_md.set_pbc(True)
water_md.calc = calculator

MaxwellBoltzmannDistribution(water_md, temperature_K=temperature_K, rng=np.random)
Stationary(water_md)
ZeroRotation(water_md)

integrator = Bussi(
    water_md,
    timestep=0.5 * ase.units.fs,
    temperature_K=temperature_K,
    taut=10.0 * ase.units.fs,
)

rdf_edges = np.linspace(0.5, 4.5, 250)
rdf_centers = 0.5 * (rdf_edges[:-1] + rdf_edges[1:])
shell_volumes = 4.0 * np.pi / 3.0 * (rdf_edges[1:] ** 3 - rdf_edges[:-1] ** 3)
oxygen_indices = [i for i, atom in enumerate(water_md) if atom.symbol == "O"]
hydrogen_indices = [i for i, atom in enumerate(water_md) if atom.symbol == "H"]
hydrogen_density = len(hydrogen_indices) / water_md.get_volume()


def oh_distance_distribution(atoms) -> np.ndarray:
    distances = atoms.get_all_distances(mic=True)
    distances = distances[np.ix_(oxygen_indices, hydrogen_indices)].ravel()
    histogram, _ = np.histogram(distances, bins=rdf_edges)
    return histogram / (len(oxygen_indices) * hydrogen_density * shell_volumes)


energies = []
sampled_structures = []


def collect_sample() -> None:
    sampled_structures.append(water_md.copy())
    energies.append(water_md.get_potential_energy())


integrator.attach(collect_sample, interval=2)
integrator.run(1000)

After the run we evaluate the ensemble energy of each frame and compute its O–H RDF.

In [ ]:
energies = np.asarray(energies)
ensemble_outputs = {"energy_ensemble": ModelOutput()}
ensemble_energies = np.array(
    [
        calculator.run_model(atoms, ensemble_outputs)["energy_ensemble"][0]
        .values.detach()
        .numpy()
        .squeeze()
        for atoms in sampled_structures
    ]
)
oh_rdfs = np.array([oh_distance_distribution(atoms) for atoms in sampled_structures])

### Exercise 3a — the reweighting weights

For a canonical ensemble at temperature $T = 1/(\beta k_\mathrm{B})$, the reweighted
average of an observable $a$ under member potential $V^{(i)}$ is
([Imbalzano *et al.*, 2021](https://doi.org/10.1063/5.0036522), Eq. 22):

$$\langle a \rangle_{V^{(i)}} =
  \frac{\langle w^{(i)}\, a \rangle_{\bar{V}}}{\langle w^{(i)} \rangle_{\bar{V}}},
  \qquad
  w^{(i)}(\mathbf{x}_t) =
    \exp\!\left(-\beta\left[V^{(i)}(\mathbf{x}_t) - \bar{V}(\mathbf{x}_t)\right]\right).$$

A frame is **up-weighted** for a member that assigns it *lower* energy than the mean
potential, and down-weighted otherwise.

**Your task.** Fill in the (unnormalized) log-weights $h = -\beta\,(V^{(i)} - \bar{V})$.
`ensemble_energies` has shape `(n_frames, n_members)`; `energies` is $\bar V$ per frame,
shape `(n_frames,)`.
*Hint:* broadcast the mean potential with `energies[:, None]`.

In [ ]:
beta = 1.0 / (ase.units.kB * temperature_K)

# TODO (Exercise 3a): log-weight h = -beta * (V_i - Vbar) for every (frame, member).
log_weights = None  # <-- your code using `ensemble_energies` and `energies`

In [ ]:
# Reference solution (Exercise 3a).
if log_weights is None:
    log_weights = -beta * (ensemble_energies - energies[:, None])

In [ ]:
# Stabilize (subtract per-frame max), exponentiate, normalize, then reweight.
log_weights -= log_weights.max(axis=0, keepdims=True)
weights = np.exp(log_weights)
weights /= weights.sum(axis=0, keepdims=True)

reweighted_rdf_by_member = weights.T @ oh_rdfs

reweighted_rdf_mean = reweighted_rdf_by_member.mean(axis=0)
reweighted_rdf_std = reweighted_rdf_by_member.std(axis=0, ddof=1)

The reweighted ensemble mean and its $\pm 1\sigma$ band:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

ax.plot(rdf_centers, reweighted_rdf_mean, label="Reweighted ensemble mean", lw=1.5)
ax.fill_between(
    rdf_centers,
    reweighted_rdf_mean - reweighted_rdf_std,
    reweighted_rdf_mean + reweighted_rdf_std,
    alpha=0.25,
    label=r"$\pm 1\sigma$",
)
ax.set(
    xlabel="O-H distance / Å",
    ylabel="Probability density",
    title="Ensemble mean and spread",
    ylim=(0.0, 3.0),
)
ax.grid()
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

## Uncertainty propagation with i-PI

The same propagation can be done with [i-PI](https://ipi-code.org), which supports more
advanced MD and handles the reweighting as post-processing. Here we look at the H–H and
O–O RDFs of the same 32-water box.

### Cumulant Expansion Approximation (CEA)

To avoid the exponential growth of the statistical error with system size, i-PI uses the
**CEA** by default ([Imbalzano *et al.*, 2021](https://doi.org/10.1063/5.0036522),
Eq. 24), replacing the exponential weight with its first-order expansion in
$h^{(i)} = \beta(V^{(i)} - \bar{V})$:

$$\langle a \rangle_{V^{(i)}} \approx \langle a \rangle_{\bar{V}}
  - \beta\left[\langle a\,(V^{(i)} - \bar{V}) \rangle_{\bar{V}}
  - \langle a \rangle_{\bar{V}}\,\langle V^{(i)} - \bar{V} \rangle_{\bar{V}}\right].$$

The CEA is statistically stable (its error grows only linearly with system size) and
consistent to first order; it is reliable when the committee members agree closely
($\sigma^2(h^{(i)}) \ll 1$). For strongly non-linear observables it must be used with care
(see Kellner & Ceriotti, 2024, §4.5).

The driver's `uncertainty_threshold` warns when an atomic-energy uncertainty exceeds the
given value (eV/atom); here it is set to 100 eV/atom (never reached) to keep the printout
clean.

In [ ]:
# Load configuration and start the simulation.
with open("data/h2o-32.xml") as f:
    xml_input = f.read()

# Print the relevant section of the input file.
print(xml_input[:883][-334:])

sim = InteractiveSimulation(xml_input)

Run the simulation. **NB:** increase the step count (e.g. to 10000) for better
estimates; here we keep it short.

In [ ]:
sim.run(2000)

Load the trajectory and compute the per-frame H–H and O–O RDFs. (ASE uses an
unusual normalization for partial RDFs; we apply a light smoothing because the short run
gives few frames.)

In [ ]:
frames = ase.io.read("h2o-32.pos_0.extxyz", ":")

# Sanity check: only water (H = 1, O = 8).
assert set(frames[0].numbers.tolist()) == set([1, 8])

num_bins = 90
rdfs_hh = []
rdfs_oo = []
xs = None
for atoms in frames:
    # H-H
    bins, xs = ase.geometry.rdf.get_rdf(atoms, 4.5, num_bins, elements=[1, 1])
    bins[2:-2] = (
        bins[:-4] * 0.1
        + bins[1:-3] * 0.2
        + bins[2:-2] * 0.4
        + bins[3:-1] * 0.2
        + bins[4:] * 0.1
    )
    rdfs_hh.append(bins)

    # O-O
    bins, xs = ase.geometry.rdf.get_rdf(atoms, 4.5, num_bins, elements=[8, 8])
    bins[2:-2] = (
        bins[:-4] * 0.1
        + bins[1:-3] * 0.2
        + bins[2:-2] * 0.4
        + bins[3:-1] * 0.2
        + bins[4:] * 0.1
    )
    rdfs_oo.append(bins)

rdfs_hh = np.stack(rdfs_hh, axis=0)
rdfs_oo = np.stack(rdfs_oo, axis=0)

Run i-PI's committee re-weighting utility as a post-processing step.

In [ ]:
# Save RDFs so i-PI can read them.
np.savetxt("h2o-32_rdfs_h-h.txt", rdfs_hh)
np.savetxt("h2o-32_rdfs_o-o.txt", rdfs_oo)

# Run the re-weighting tool for H-H and O-O.
for ty in ["h-h", "o-o"]:
    infile = f"h2o-32_rdfs_{ty}.txt"
    outfile = f"h2o-32_rdfs_{ty}_reweighted.txt"
    cmd = (
        f"i-pi-committee-reweight h2o-32.committee_pot_0 {infile} --input"
        " data/h2o-32.xml"
    )
    print("Executing command:", "\t" + cmd, sep="\n")
    cmd = cmd.split()
    with open(outfile, "w") as out:
        process = subprocess.run(cmd, stdout=out)

### Exercise 3b — the uncertainty band

The reweighted file stores the mean RDF in column 0 and its propagated standard deviation
in column 1. We will shade a $\pm 1\sigma$ band around the mean.

**Your task.** Complete `sigma_band(mus, std)` so it returns a `(lower, upper)` tuple of
arrays — the band edges passed to `fill_between`.
*Hint:* `lower = mus - std`, `upper = mus + std`.

In [ ]:
def sigma_band(mus, std):
    # TODO (Exercise 3b): return the (lower, upper) +/- 1 sigma band.
    return None  # <-- replace, e.g. (mus - std, mus + std)

In [ ]:
# Reference solution (Exercise 3b).
if sigma_band(np.zeros(1), np.ones(1)) is None:

    def sigma_band(mus, std):
        return (mus - std, mus + std)

In [ ]:
rdfs_hh_reweighted = np.loadtxt("h2o-32_rdfs_h-h_reweighted.txt")
rdfs_oo_reweighted = np.loadtxt("h2o-32_rdfs_o-o_reweighted.txt")

rdfs_hh_reweighted_mu = rdfs_hh_reweighted[:, 0]
rdfs_hh_reweighted_err = rdfs_hh_reweighted[:, 1]

rdfs_oo_reweighted_mu = rdfs_oo_reweighted[:, 0]
rdfs_oo_reweighted_err = rdfs_oo_reweighted[:, 1]

fig, axs = plt.subplots(figsize=(6, 3), sharey=True, ncols=2, constrained_layout=True)
for title, ax, mus, std, xlim in [
    ("H-H", axs[0], rdfs_hh_reweighted_mu, rdfs_hh_reweighted_err, (1.0, 4.5)),
    ("O-O", axs[1], rdfs_oo_reweighted_mu, rdfs_oo_reweighted_err, (2.0, 4.5)),
]:
    ylabel = "RDF" if title == "H-H" else None
    ax.set(title=title, xlabel="Distance", ylabel=ylabel, xlim=xlim, ylim=(-0.1, 3.7))
    ax.grid()
    ax.plot(xs, mus, label="Mean", lw=0.5)
    ax.fill_between(xs, *sigma_band(mus, std), alpha=0.5, label=r"$\pm 1 \sigma$")
    ax.legend()
plt.show()

### Comparison to experiment

Finally we compare the reweighted O–O RDF to experiment (Soper, 2000, *Chem. Phys.*
**258**, [doi](https://doi.org/10.1016/S0301-0104(00)00179-8)). Think about what is still
missing to make the two curves coincide.

In [ ]:
experimental_oo = np.loadtxt("data/OO.csv", delimiter=",")

fig, ax = plt.subplots(figsize=(5, 4))
ax.set(
    title="O-O Radial Distribution Function",
    xlabel="Distance / $\\mathrm{\\AA}$",
    ylabel="RDF",
    xlim=(2.0, 4.5),
    ylim=(-0.1, 3.7),
)
ax.grid()

# ML prediction (mean and std).
ax.plot(xs, rdfs_oo_reweighted_mu, label="PET-MAD (Mean)", lw=1.5)
ax.fill_between(
    xs,
    rdfs_oo_reweighted_mu - rdfs_oo_reweighted_err,
    rdfs_oo_reweighted_mu + rdfs_oo_reweighted_err,
    alpha=0.5,
    label=r"PET-MAD ($\pm 1 \sigma$)",
)

# Experimental data.
ax.plot(experimental_oo[:, 0], experimental_oo[:, 1], "k--", label="Experiment", lw=1.5)

ax.legend()
fig.tight_layout()
plt.show()

## Eliminating error sources with converged simulations

Several error sources were introduced *by design* to keep this tutorial light: a short
simulation time, a box with a small number of H2O molecules, the neglect of nuclear quantum effects, and the intrinsic
accuracy of the base model. The two easiest to address are **simulation length** and
**model size**. The precomputed results below used **500 ps (1,000,000 steps)** of NVT
dynamics for both the PET-MAD XS and S models; the bands are the i-PI CEA-reweighted
$\pm 1\sigma$ across the committee.

To switch from XS to S yourself, update the model download call:

```python
upet.save_upet(model="pet-mad", size="s", version="1.5.0", output=model_path)
```

and the `model` key in the `ffdirect` parameters block of `data/h2o-32.xml`:

```xml
{ template:data/h2o-32.xyz, model:models/pet-mad-s-v1.5.0.pt, ... }
```

On an HPC cluster you can run i-PI standalone (this input uses `ffdirect`, so no separate
driver is needed). Add `<total_steps>` inside the `<motion>` block of `data/h2o-32.xml`:

```xml
<motion mode='dynamics'>
  <total_steps> 1000000 </total_steps>
  ...
</motion>
```

then submit with:

```bash
i-pi data/h2o-32.xml
```

We ship precomputed O–O RDFs for both XS and S (500 ps trajectories); load and plot
them directly against experiment.

In [ ]:
bins_xs = np.load("data/MAD_XS_precompute/rdf_bins.npy")
mean_xs = np.load("data/MAD_XS_precompute/rdf_mean_rdf_o-o.npy")
rew_std_xs = np.load("data/MAD_XS_precompute/rdf_reweighted_std_o-o.npy")

bins_s = np.load("data/MAD_S_precompute/rdf_bins.npy")
mean_s = np.load("data/MAD_S_precompute/rdf_mean_rdf_o-o.npy")
rew_std_s = np.load("data/MAD_S_precompute/rdf_reweighted_std_o-o.npy")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.set(xlim=(2.0, 4.5), ylim=(-0.1, 3.7))
ax.tick_params(labelsize=12)
ax.grid()

ax.plot(bins_xs, mean_xs, label="PET-MAD XS", lw=1.5, color="C0")
ax.fill_between(
    bins_xs,
    mean_xs - rew_std_xs,
    mean_xs + rew_std_xs,
    alpha=0.3,
    color="C0",
    label="PET-MAD XS ($\\pm 1\\sigma$)",
)

ax.plot(bins_s, mean_s, label="PET-MAD S", lw=1.5, color="C1")
ax.fill_between(
    bins_s,
    mean_s - rew_std_s,
    mean_s + rew_std_s,
    alpha=0.3,
    color="C1",
    label="PET-MAD S ($\\pm 1\\sigma$)",
)

ax.plot(
    experimental_oo[:, 0],
    experimental_oo[:, 1],
    "k--",
    lw=1.5,
    label="Experiment",
)

ax.set_xlabel("Distance / $\\mathrm{\\AA}$", fontsize=13)
ax.set_ylabel("RDF", fontsize=13)
ax.set_title("O-O RDF: model comparison (500 ps)", fontsize=13)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

### Takeaways

The O–O RDF from the **S** model agrees better with experiment than **XS**, consistent
with its higher accuracy on reference data, and its uncertainty band is slightly narrower
— yet still elevated. That suggests liquid water is not perfectly described even by S, and
further confidence could come from fine-tuning.

**Questions to think about**

- *Exercise 1.* Why are larger structures more uncertain? How could you use uncertainties to build better datasets?
- *Exercise 2.* Is it justified to relax with the ensemble *mean* and then evaluate
  ensemble energies on that single structure? Why does the large spread of the bulk energy
  mostly cancel in the formation energy?
- *Exercise 3.* The O–O RDF uncertainty stays sizeable even for long trajectories. Is this
  a *sampling* or a *model* problem — and how would you test which?

### Advanced exercise: generating the LLPR uncertainty estimator yourself

The LLPR uncertainty estimator is a powerful tool, but it requires an additional post-processing step.

If you are interested in how it works under the hood and want to understand the input file, we have prepared an advanced tutorial in the Atomistic Cookbook repository:

[LLPR from scratch](https://atomistic-cookbook.org/examples/llpr-from-scratch/llpr-from-scratch.html)